# 04b. 경량 모델 크기·속도 측정

- 담당: **④ 경량화·통합** (이 파일은 담당자만 수정)
- 목적: 기획서 4장 "왜 경량 모델인가" 근거. 성능 차이 0.02 이하인 경량 모델이 고성능 모델보다 가볍고 빠른지 비교
- 비교 대상: 경량(로지스틱 회귀 E) · 고성능(③ XGBoost E depth3) · 앙상블(두 모델 확률 50:50)
- 비교 기준: 모델 단독 — 모델 크기, 전처리된 데이터로 잰 모델 계산 시간 (두 모델이 같은 E 피처 전처리를 쓰므로)
- 참고: 전처리 포함 전체 시간 (서비스 시나리오용)
- 측정 방법: 세 모델 모두 같은 전처리 결과를 입력으로, 워밍업 1회 후 반복 측정의 중앙값 (1명 20회, 500명 5회). 앙상블은 확률 평균까지 포함
- 측정 조건: 같은 컴퓨터(CPU), train 500명 사용 (test 사용 안 함). OS·python·pandas 버전은 2번 섹션 출력
- 기록표에는 적지 않음 (결과는 이 노트북 출력)

> **규칙:** 전처리는 Pipeline 안에 / test는 보지 않기 / 점수는 `run_cv()`로 / 결과는 `log_result()`로 기록 / 자세한 규칙은 `docs/coding_rules.md`

## 0. 환경 설정 (수정하지 말고 그대로 실행)
Colab은 구글 드라이브 `내 드라이브`에 레포 폴더(`ah07-cancer-subtype`, 안에 `data/` 포함)를 두면 돼요.

In [19]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/drive/MyDrive/ah07-cancer-subtype"  # 드라이브에 둔 레포 폴더
else:
    REPO_DIR = os.path.abspath(os.path.join("..", ".."))  # notebooks/이니셜_역할/ 폴더 기준

# 경로 확인: 여기서 멈추면 폴더 위치나 이름을 확인해 주세요
assert os.path.isdir(os.path.join(REPO_DIR, "src")), (
    f"src 폴더를 찾을 수 없어요: {REPO_DIR}\n"
    "→ 드라이브 '내 드라이브' 바로 아래에 'ah07-cancer-subtype' 폴더가 있는지, "
    "ZIP을 풀 때 폴더가 두 번 겹치지 않았는지(ah07-cancer-subtype/ah07-cancer-subtype-dev) 확인")
assert os.path.isfile(os.path.join(REPO_DIR, "data", "train.csv")), (
    "data/train.csv 가 없어요 → 데이콘 csv 3개를 data/ 폴더에 넣어 주세요")
sys.path.insert(0, REPO_DIR)

from src.config import SEED, DATA_DIR, set_seed
set_seed(SEED)
print("REPO_DIR:", REPO_DIR, "| DATA_DIR:", DATA_DIR)

REPO_DIR: /Users/admin/Documents/dacon_hackerton/4team_tuna/ah07-cancer-subtype | DATA_DIR: /Users/admin/Documents/dacon_hackerton/4team_tuna/ah07-cancer-subtype/data


In [20]:
import platform
import numpy as np, pandas as pd, sklearn
from sklearn.pipeline import Pipeline

from src.data_loader import load_data, split_xy, encode_target
from src.preprocessing import basic_transformer, binarize_mutations
from src.features import mutation_count
from src.validation import get_folds, run_cv, balanced_weights
from src.evaluation import macro_f1, per_class_f1, log_result
from src.prediction import make_submission

print("OS:", platform.platform(), "| python", platform.python_version(),
      "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

OS: macOS-27.0.1-arm64-arm-64bit-Mach-O | python 3.13.18 | pandas 3.1.1 | scikit-learn 1.9.1


## 1. 데이터

In [21]:
train, test = load_data(DATA_DIR)   # test는 형식 확인과 최종 추론에만
X, y_raw = split_xy(train)
y, le = encode_target(y_raw)
folds = get_folds(y)                 # 팀 공통 fold
print("train", train.shape, "| test", test.shape, "| classes", len(le.classes_))

train (6201, 4386) | test (2546, 4385) | classes 26


## 2. 측정 준비 — 크기·시간 측정 함수

In [22]:
import io, time, joblib
from scipy.sparse import csr_matrix
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from src.features import (mutation_count, mutation_type_counts,
                          amino_acid_pair_counts, gene_truncation_flags)

print("측정 환경:", platform.platform(), "| python", platform.python_version(),
      "| pandas", pd.__version__)

def size_mb(*models):
    """저장했을 때의 크기(MB). 모델이 여러 개면 함께 저장한 크기."""
    buffer = io.BytesIO()
    joblib.dump(models if len(models) > 1 else models[0], buffer)
    return round(buffer.getbuffer().nbytes / 1024**2, 2)

def median_ms(fn, n_repeat):
    """워밍업 1회 후 n_repeat번 실행 시간의 중앙값(ms)."""
    fn()
    times = []
    for _ in range(n_repeat):
        start = time.perf_counter()
        fn()
        times.append((time.perf_counter() - start) * 1000)
    return round(float(np.median(times)), 3)

X_timing = X.iloc[:500]   # 시간 측정용 샘플 (train, test 사용 안 함)

측정 환경: macOS-27.0.1-arm64-arm-64bit-Mach-O | python 3.13.18 | pandas 3.1.1


## 3. 모델 학습 — 같은 E 피처로 경량·고성능 학습
E 피처: ② 02_features의 make_features_E와 같은 계산 (9,174열). 경량은 ②의 E 설정, 고성능은 ③ 04_xgb_feature_E의 depth3 설정.

In [23]:
def make_features_E(X):
    """② E 조합 피처. 한 환자 안에서만 계산."""
    return pd.concat([binarize_mutations(X),
                      np.log1p(mutation_count(X)),
                      np.log1p(mutation_type_counts(X)),
                      np.log1p(amino_acid_pair_counts(X)),
                      gene_truncation_flags(X)], axis=1)

def make_features_E_sparse(X):
    return csr_matrix(make_features_E(X).to_numpy(dtype=np.float32))

LR_E_PARAMS = dict(C=0.1, class_weight="balanced", max_iter=3000)         # ② E 설정
XGB_E_PARAMS = dict(n_estimators=250, max_depth=3, learning_rate=0.05,
                    subsample=0.8, colsample_bytree=0.8,
                    objective="multi:softprob", eval_metric="mlogloss",
                    tree_method="hist", random_state=SEED, n_jobs=-1)    # ③ depth3 설정

X_train_ready = make_features_E_sparse(X)                                # 학습용 전처리 1회
light_model = LogisticRegression(**LR_E_PARAMS).fit(X_train_ready, y)
heavy_model = XGBClassifier(**XGB_E_PARAMS).fit(X_train_ready, y)
print("학습 완료 | 피처:", X_train_ready.shape)

학습 완료 | 피처: (6201, 9174)


## 4. 모델 단독 비교 — 크기·계산 시간 (기획서 4장 표)
세 모델 모두 같은 전처리 결과(1명, 500명)를 입력으로 같은 방법으로 측정. CV Macro F1은 팀 기록표 값(그룹 K-fold).

In [24]:
X_one = make_features_E_sparse(X_timing.iloc[:1])     # 전처리된 1명
X_batch = make_features_E_sparse(X_timing)            # 전처리된 500명

def predict_ensemble(Z):
    """③ XGBoost E + 로지스틱 회귀 E 확률 50:50 평균."""
    return 0.5 * light_model.predict_proba(Z) + 0.5 * heavy_model.predict_proba(Z)

MODELS = {   # 이름: (CV Macro F1, 예측 함수, 저장할 모델)
    "경량: 로지스틱 회귀 E": (0.4919, light_model.predict_proba, (light_model,)),
    "고성능: ③ XGBoost E": (0.4870, heavy_model.predict_proba, (heavy_model,)),
    "앙상블: XGBoost E + 로지스틱 회귀 E": (0.5096, predict_ensemble, (light_model, heavy_model)),
}

rows = []
for name, (cv, predict, parts) in MODELS.items():
    rows.append({"모델": name, "CV Macro F1": cv,
                 "모델 크기(MB)": size_mb(*parts),
                 "1명 모델 계산(ms)": median_ms(lambda: predict(X_one), 20),
                 "500명 모델 계산(ms)": median_ms(lambda: predict(X_batch), 5)})
model_table = pd.DataFrame(rows)
model_table

,모델,CV Macro F1,모델 크기(MB),1명 모델 계산(ms),500명 모델 계산(ms)
0,경량: 로지스틱 회귀 E,0.4919,0.91,0.062,0.516
1,고성능: ③ XGBoost E,0.4870,6.74,0.779,8.421
2,앙상블: XGBoost E + 로지스틱 회귀 E,0.5096,7.66,0.963,9.694


## 5. 참고 — 전처리 포함 전체 시간 (서비스 시나리오)
원본 데이터(유전자 표기 글자)를 E 피처로 바꾸는 시간 + 4번의 모델 계산 시간.

In [25]:
prep_one_ms = median_ms(lambda: make_features_E_sparse(X_timing.iloc[:1]), 20)
prep_batch_ms = median_ms(lambda: make_features_E_sparse(X_timing), 5)
print("전처리 | 1명:", prep_one_ms, "ms | 500명:", prep_batch_ms, "ms")

total_table = model_table[["모델"]].copy()
total_table["1명 전체(ms)"] = (prep_one_ms + model_table["1명 모델 계산(ms)"]).round(1)
total_table["500명 전체(ms)"] = (prep_batch_ms + model_table["500명 모델 계산(ms)"]).round(1)
total_table

전처리 | 1명: 945.095 ms | 500명: 3642.023 ms


,모델,1명 전체(ms),500명 전체(ms)
0,경량: 로지스틱 회귀 E,945.2,3642.5
1,고성능: ③ XGBoost E,945.9,3650.4
2,앙상블: XGBoost E + 로지스틱 회귀 E,946.1,3651.7


## 6. 요약 — 경량 vs 고성능 배수, 앙상블 대비 성능 차이

In [26]:
light, heavy = model_table.iloc[0], model_table.iloc[1]
print("모델 크기:", round(heavy["모델 크기(MB)"] / light["모델 크기(MB)"], 1), "배 차이")
print("1명 모델 계산:", round(heavy["1명 모델 계산(ms)"] / light["1명 모델 계산(ms)"], 1), "배 차이")
print("500명 모델 계산:", round(heavy["500명 모델 계산(ms)"] / light["500명 모델 계산(ms)"], 1), "배 차이")
print("앙상블 대비 경량 성능 차이:", round(0.5096 - 0.4919, 4), "(채택 기준 0.02 이하)")

모델 크기: 7.4 배 차이
1명 모델 계산: 12.6 배 차이
500명 모델 계산: 16.3 배 차이
앙상블 대비 경량 성능 차이: 0.0177 (채택 기준 0.02 이하)
